# Unified Sub-100M Trainer (Variant C or E)

This notebook consolidates sub-100M training for Variant C and Variant E into one pipeline.

What you get:
- One variant switch (`c` or `e`) for architecture selection.
- Unified ~40M profiles for fair head-to-head comparison.
- Manifest auto-build from NPZ packs.
- Resume support, consistent outputs, and summary reporting.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import subprocess
import sys

REQUIRED_PIP_PACKAGES = {
    "miditok": "miditok",
    "pretty_midi": "pretty_midi",
    "ncps": "ncps",
    "safetensors": "safetensors",
    "matplotlib": "matplotlib",
    "huggingface_hub": "huggingface_hub",
}

def ensure_runtime_dependencies() -> None:
    missing_specs = [
        spec
        for module_name, spec in REQUIRED_PIP_PACKAGES.items()
        if importlib.util.find_spec(module_name) is None
    ]
    if not missing_specs:
        print("Runtime dependencies ready.")
        return

    print(f"Installing missing package(s): {missing_specs}")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing_specs,
        ]
    )

def can_write_to(path: Path) -> bool:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".copilot_write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except Exception:
        return False

def find_project_root() -> Path:
    anchor = Path("training/ablation_runner.py")
    start = Path.cwd().resolve()
    probes = [start]
    if start.name.lower() == "notebooks":
        probes.append(start.parent)

    kaggle_hints = [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
        Path("/kaggle/working/piano_midi_model"),
        Path("/kaggle/input/piano_midi_model"),
        Path("/kaggle/input/piano-midi-model"),
        Path("/kaggle/input/midi-ai-piano-model"),
    ]
    probes.extend(path for path in kaggle_hints if path.exists())

    seen = set()
    for probe in probes:
        if not probe.exists():
            continue
        key = str(probe)
        if key in seen:
            continue
        seen.add(key)
        for parent in [probe, *probe.parents]:
            if (parent / anchor).exists() and (parent / "training" / "__init__.py").exists():
                return parent

    raise FileNotFoundError(f"Could not locate project root containing {anchor}")

def resolve_output_base(project_root: Path) -> Path:
    candidates = []
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists():
        candidates.append(kaggle_working)
    candidates.append(project_root)

    for candidate in candidates:
        if can_write_to(candidate):
            return candidate
    raise OSError("Could not find writable output base.")

ensure_runtime_dependencies()
PROJECT_ROOT = find_project_root()
OUTPUT_BASE = resolve_output_base(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

print(f"Project root: {PROJECT_ROOT}")
print(f"Output base: {OUTPUT_BASE}")

In [ ]:
import numpy as np
import torch

VARIANT = "e"  # "c" or "e"
SIZE_PROFILE = "40m"
MAX_PIECES = 100_000
EPOCHS = 20
SEED = 42

SEED_LENGTH = 512
CONTINUATION_LENGTH = 1536
MAX_SEQUENCE_LENGTH = SEED_LENGTH + CONTINUATION_LENGTH

TARGET_EFFECTIVE_BATCH = 32
BATCH_PER_GPU = 2
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.1
WARMUP_STEPS = 0
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1
MAX_GRAD_NORM = 1.0
LOG_EVERY_N_STEPS = 20
SAVE_EVERY_N_EPOCHS = 5
KEEP_EVERY_N_EPOCHS = 10
MAX_CHECKPOINTS = 8

AUTO_RESUME = True
RESUME_FROM_CHECKPOINT = ""
RESUME_MODE = "remaining"

SEED_MIDI = ""
GENERATION_MAX_NEW_TOKENS = 8192
GENERATION_CONTINUATION_SECONDS = 120.0
GENERATION_TEMPERATURE = 0.9
GENERATION_TOP_P = 0.95
GENERATION_TOP_K = 50
GENERATION_REPETITION_PENALTY = 1.1
GENERATION_REPETITION_WINDOW = 64
GENERATION_MIN_TOKENS_TO_KEEP = 3
GENERATION_MAX_CONSECUTIVE_ZERO_DELTAS = 12

ALLOW_FALLBACK_GDN = False
ALLOW_GDN_DATA_PARALLEL = False
RUN_TRAINING = False

FORCE_REBUILD_MANIFEST = False
PRETOKENIZED_ROOT_CANDIDATES = [
    PROJECT_ROOT / "processed",
    Path("/kaggle/input/godzilla-tokenized-100k/tokenized"),
    Path("/kaggle/input/godzilla-tokenized-100k"),
    Path("/kaggle/input/godzilla-tokenized-15k/tokenized"),
    Path("/kaggle/input/godzilla-tokenized-15k"),
]

ARCHITECTURE_PROFILES = {
    "c": {
        "40m": {
            "d_model": 512,
            "n_layers": 12,
            "num_attention_heads": 8,
            "ffn_expansion": 4,
            "target_params": 40_000_000,
        }
    },
    "e": {
        "40m": {
            "d_model": 768,
            "n_layers": 13,
            "attention_every_n_layers": 2,
            "gdn_inner_ratio": 0.5,
            "gdn_num_heads": 4,
            "gqa_num_heads": 8,
            "gqa_groups": 4,
            "target_params": 40_000_000,
        }
    },
}

if VARIANT not in ARCHITECTURE_PROFILES:
    raise ValueError(f"Unsupported VARIANT={VARIANT}. Use c or e.")

GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 1
BATCH_SIZE = max(1, GPU_COUNT * BATCH_PER_GPU) if GPU_COUNT > 1 else 1
GRAD_ACCUM_STEPS = max(1, int(np.ceil(float(TARGET_EFFECTIVE_BATCH) / float(BATCH_SIZE))))
NUM_WORKERS = max(0, int(GPU_COUNT) * 2)

ENABLE_DATA_PARALLEL_C = GPU_COUNT > 1
ENABLE_DATA_PARALLEL_E = GPU_COUNT > 1

run_tag = f"sub100m_{VARIANT}_{SIZE_PROFILE}_{MAX_PIECES // 1000}k"
OUTPUT_DIR = OUTPUT_BASE / "outputs" / run_tag
PRETOKENIZED_MANIFEST = OUTPUT_DIR / "processed_pretokenized" / "manifest.json"
RESULT_JSON = OUTPUT_DIR / f"variant_{VARIANT}_{SIZE_PROFILE}_result.json"

print(f"Variant: {VARIANT} | Profile: {SIZE_PROFILE}")
print(f"Max pieces: {MAX_PIECES:,}")
print(f"Batch size: {BATCH_SIZE} | Grad accumulation: {GRAD_ACCUM_STEPS}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
import json

def discover_npz_parent(path: Path):
    if not path.exists():
        return None
    hit = next(path.rglob("*.npz"), None)
    return hit.parent if hit is not None else None

def find_npz_root(candidates) -> Path:
    for candidate in candidates:
        resolved = discover_npz_parent(Path(candidate))
        if resolved is not None:
            return resolved

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for folder in kaggle_input.iterdir():
            if not folder.is_dir():
                continue
            resolved = discover_npz_parent(folder)
            if resolved is not None:
                return resolved

    raise FileNotFoundError("Could not locate tokenized NPZ files.")

def build_manifest_from_npz(npz_root: Path, manifest_path: Path) -> int:
    npz_files = sorted(npz_root.rglob("*.npz"))
    if not npz_files:
        raise FileNotFoundError(f"No .npz files found under: {npz_root}")

    manifest = []
    skipped = 0
    for npz_path in npz_files:
        try:
            with np.load(npz_path, allow_pickle=False) as pack:
                if "tokens" not in pack.files:
                    skipped += 1
                    continue
                token_len = int(np.asarray(pack["tokens"]).shape[0])
        except Exception:
            skipped += 1
            continue

        manifest.append({
            "md5": npz_path.stem,
            "npz_path": str(npz_path),
            "length": token_len,
            "source_path": str(npz_path.parent),
        })

    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Manifest built: {manifest_path} | entries={len(manifest)} | skipped={skipped}")
    return len(manifest)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRETOKENIZED_ROOT = find_npz_root(PRETOKENIZED_ROOT_CANDIDATES)
if FORCE_REBUILD_MANIFEST or not PRETOKENIZED_MANIFEST.exists():
    _ = build_manifest_from_npz(PRETOKENIZED_ROOT, PRETOKENIZED_MANIFEST)

print(f"Pretokenized root: {PRETOKENIZED_ROOT}")
print(f"Manifest path: {PRETOKENIZED_MANIFEST}")

In [ ]:
import json
import math
from config import DataConfig, TrainConfig
from data.tokenizer_custom import CustomDeltaTokenizer
from model.variant_c import VariantCConfig, VariantCModel
from model.variant_e import VariantEConfig, VariantEModel
from training.ablation_runner import (
    _build_dataloaders,
    _load_pretokenized_manifest,
    _run_variant,
    _set_global_seed,
    _train_val_split,
    _variant_backend_status,
)

def resolve_divisible_heads(width: int, requested_heads: int) -> int:
    heads = max(1, min(int(requested_heads), int(width)))
    while heads > 1 and (int(width) % heads) != 0:
        heads -= 1
    return max(1, heads)

def build_variant_model(variant: str, vocab_size: int, max_sequence_length: int):
    profile = dict(ARCHITECTURE_PROFILES[variant][SIZE_PROFILE])

    if variant == "c":
        heads = resolve_divisible_heads(int(profile["d_model"]), int(profile["num_attention_heads"]))
        model = VariantCModel(
            VariantCConfig(
                vocab_size=int(vocab_size),
                d_model=int(profile["d_model"]),
                n_layers=int(profile["n_layers"]),
                max_sequence_length=int(max_sequence_length),
                num_attention_heads=int(heads),
                ffn_expansion=int(profile["ffn_expansion"]),
            )
        )
        shape = {
            "d_model": int(profile["d_model"]),
            "n_layers": int(profile["n_layers"]),
            "num_attention_heads": int(heads),
            "ffn_expansion": int(profile["ffn_expansion"]),
        }
        return model, shape

    d_model = int(profile["d_model"])
    gdn_inner_dim = max(128, int(round(float(d_model) * float(profile["gdn_inner_ratio"]))))
    gdn_heads = resolve_divisible_heads(gdn_inner_dim, int(profile["gdn_num_heads"]))
    gqa_heads = resolve_divisible_heads(d_model, int(profile["gqa_num_heads"]))
    gqa_groups = max(1, min(int(profile["gqa_groups"]), gqa_heads))
    while gqa_groups > 1 and (gqa_heads % gqa_groups) != 0:
        gqa_groups -= 1

    model = VariantEModel(
        VariantEConfig(
            vocab_size=int(vocab_size),
            d_model=d_model,
            n_layers=int(profile["n_layers"]),
            max_sequence_length=int(max_sequence_length),
            gdn_inner_dim=int(gdn_inner_dim),
            gdn_num_heads=int(gdn_heads),
            gqa_num_heads=int(gqa_heads),
            gqa_groups=int(gqa_groups),
            attention_every_n_layers=int(profile["attention_every_n_layers"]),
        )
    )
    shape = {
        "d_model": int(d_model),
        "n_layers": int(profile["n_layers"]),
        "attention_every_n_layers": int(profile["attention_every_n_layers"]),
        "gdn_inner_dim": int(gdn_inner_dim),
        "gdn_num_heads": int(gdn_heads),
        "gqa_num_heads": int(gqa_heads),
        "gqa_groups": int(gqa_groups),
    }
    return model, shape

def pick_resume_checkpoint(default_checkpoint_dir: Path, raw_value: str, auto_resume: bool):
    value = str(raw_value).strip()

    def pick_from_dir(dir_path: Path):
        candidates = [
            dir_path / "latest_state.pt",
            dir_path / "latest.safetensors",
            dir_path / "best_state.pt",
            dir_path / "best.safetensors",
        ]
        return next((p for p in candidates if p.exists()), None)

    if value:
        candidate = Path(value).expanduser()
        if candidate.is_file():
            return candidate
        if candidate.is_dir():
            found = pick_from_dir(candidate)
            if found is None:
                raise FileNotFoundError(f"No checkpoint files found in directory {candidate.resolve()}")
            return found
        raise FileNotFoundError(f"Resume checkpoint not found: {candidate.resolve()}")

    if auto_resume:
        return pick_from_dir(default_checkpoint_dir)
    return None

_set_global_seed(int(SEED))
tokenizer = CustomDeltaTokenizer(include_special_tokens=False)

print("Approximate parameter counts for unified 40M profiles:")
for variant_key in ("c", "e"):
    check_model, _ = build_variant_model(variant_key, int(tokenizer.vocab_size), int(MAX_SEQUENCE_LENGTH))
    check_params = int(sum(p.numel() for p in check_model.parameters()))
    print(f"  variant_{variant_key}: {check_params:,} ({check_params / 1e6:.2f}M)")

manifest = _load_pretokenized_manifest(
    manifest_path=Path(PRETOKENIZED_MANIFEST),
    pretokenized_root=Path(PRETOKENIZED_ROOT),
    max_pieces=int(max(0, MAX_PIECES)),
    min_required_tokens=int(MAX_SEQUENCE_LENGTH),
)

processed_dir = OUTPUT_DIR / "processed_pretokenized"
processed_dir.mkdir(parents=True, exist_ok=True)
manifest_out = processed_dir / "manifest.json"
manifest_out.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

train_manifest, val_manifest = _train_val_split(
    manifest=manifest,
    seed=int(SEED),
    val_fraction=0.1,
)

data_cfg = DataConfig(
    tokenizer_path=str(OUTPUT_DIR / "custom_tokenizer.json"),
    processed_path=str(processed_dir),
    vocab_size=int(tokenizer.vocab_size),
    tokenization_strategy="custom_delta",
    seed_length=int(SEED_LENGTH),
    continuation_length=int(CONTINUATION_LENGTH),
    max_sequence_length=int(MAX_SEQUENCE_LENGTH),
    use_continuous_time=True,
    time_feature_fallback_step_seconds=0.1,
)
tokenizer.save(data_cfg.tokenizer_path)

probe_cfg = TrainConfig(
    batch_size=int(BATCH_SIZE),
    grad_accumulation_steps=int(GRAD_ACCUM_STEPS),
    learning_rate=float(LEARNING_RATE),
    lr_schedule="cosine",
    min_lr_ratio=float(MIN_LR_RATIO),
    weight_decay=float(WEIGHT_DECAY),
    label_smoothing=float(LABEL_SMOOTHING),
    max_epochs=1,
    warmup_steps=1,
    max_grad_norm=float(MAX_GRAD_NORM),
    save_every_n_epochs=1,
    keep_every_n_epochs=1,
    max_checkpoints=1,
    checkpoint_dir=str(OUTPUT_DIR / "checkpoints" / "_probe"),
    use_wandb=False,
    seed=int(SEED),
    device="auto",
    val_generation_check=False,
)
setattr(probe_cfg, "_force_num_workers", int(max(0, NUM_WORKERS)))
setattr(probe_cfg, "_enable_data_parallel", False)

probe_train_loader, _ = _build_dataloaders(
    train_manifest=train_manifest,
    val_manifest=val_manifest,
    data_cfg=data_cfg,
    train_cfg=probe_cfg,
    seed=int(SEED),
)

steps_per_epoch = max(
    1,
    math.ceil(len(probe_train_loader) / float(max(1, int(GRAD_ACCUM_STEPS)))),
)
total_steps = max(1, int(steps_per_epoch) * int(EPOCHS))
warmup_steps = int(max(1, WARMUP_STEPS)) if int(WARMUP_STEPS) > 0 else int(max(1, round(float(WARMUP_RATIO) * float(total_steps))))

model, model_shape = build_variant_model(VARIANT, int(tokenizer.vocab_size), int(MAX_SEQUENCE_LENGTH))
params = int(sum(p.numel() for p in model.parameters()))
backend_status = _variant_backend_status(model)

if VARIANT == "e" and backend_status.get("gdn_using_fallback", False) and not bool(ALLOW_FALLBACK_GDN):
    raise RuntimeError(
        "Variant E is using fallback GDN. Install flash-linear-attention or set ALLOW_FALLBACK_GDN=True."
    )

checkpoint_dir = OUTPUT_DIR / "checkpoints" / f"variant_{VARIANT}_{SIZE_PROFILE}"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
resume_checkpoint = pick_resume_checkpoint(checkpoint_dir, str(RESUME_FROM_CHECKPOINT), bool(AUTO_RESUME))

train_cfg = TrainConfig(
    batch_size=int(BATCH_SIZE),
    grad_accumulation_steps=int(GRAD_ACCUM_STEPS),
    learning_rate=float(LEARNING_RATE),
    lr_schedule="cosine",
    min_lr_ratio=float(MIN_LR_RATIO),
    weight_decay=float(WEIGHT_DECAY),
    label_smoothing=float(LABEL_SMOOTHING),
    max_epochs=int(EPOCHS),
    warmup_steps=int(warmup_steps),
    max_grad_norm=float(MAX_GRAD_NORM),
    save_every_n_epochs=int(max(1, SAVE_EVERY_N_EPOCHS)),
    keep_every_n_epochs=int(max(1, KEEP_EVERY_N_EPOCHS)),
    max_checkpoints=int(max(1, MAX_CHECKPOINTS)),
    checkpoint_dir=str(checkpoint_dir),
    use_wandb=False,
    seed=int(SEED),
    device="auto",
    val_generation_check=False,
)
setattr(train_cfg, "_force_num_workers", int(max(0, NUM_WORKERS)))
setattr(train_cfg, "_log_every_n_steps", int(max(1, LOG_EVERY_N_STEPS)))
if VARIANT == "c":
    setattr(train_cfg, "_enable_data_parallel", bool(ENABLE_DATA_PARALLEL_C))
else:
    setattr(train_cfg, "_enable_data_parallel", bool(ENABLE_DATA_PARALLEL_E))
    setattr(train_cfg, "_allow_gdn_data_parallel", bool(ALLOW_GDN_DATA_PARALLEL))

seed_midi_path = Path(str(SEED_MIDI)).expanduser() if str(SEED_MIDI).strip() else None
if seed_midi_path is not None and not seed_midi_path.exists():
    raise FileNotFoundError(f"Seed MIDI not found: {seed_midi_path.resolve()}")

print("Run summary:")
print(f"  variant: {VARIANT}")
print(f"  params: {params:,} ({params / 1e6:.2f}M)")
print(f"  model_shape: {model_shape}")
print(f"  backend_status: {backend_status}")
print(f"  train/val pieces: {len(train_manifest)}/{len(val_manifest)}")
print(f"  steps_per_epoch: {steps_per_epoch} | total_steps: {total_steps} | warmup_steps: {warmup_steps}")
print(f"  resume checkpoint: {resume_checkpoint if resume_checkpoint is not None else 'none'}")
print(f"  run training: {RUN_TRAINING}")

if RUN_TRAINING:
    generation_out = (
        OUTPUT_DIR / "generated" / f"variant_{VARIANT}_{SIZE_PROFILE}.mid"
        if seed_midi_path is not None
        else None
    )
    result = _run_variant(
        variant_name=f"variant_{VARIANT}",
        model=model,
        tokenizer=tokenizer,
        data_cfg=data_cfg,
        train_cfg=train_cfg,
        train_manifest=train_manifest,
        val_manifest=val_manifest,
        seed_midi=seed_midi_path,
        output_midi_path=generation_out,
        generation_max_new_tokens=int(max(1, GENERATION_MAX_NEW_TOKENS)),
        generation_continuation_seconds=float(max(1.0, GENERATION_CONTINUATION_SECONDS)),
        generation_temperature=float(max(0.1, GENERATION_TEMPERATURE)),
        generation_top_p=float(min(1.0, max(0.0, GENERATION_TOP_P))),
        generation_top_k=int(max(1, GENERATION_TOP_K)),
        generation_repetition_penalty=float(max(1.0, GENERATION_REPETITION_PENALTY)),
        generation_repetition_window=int(max(1, GENERATION_REPETITION_WINDOW)),
        generation_min_tokens_to_keep=int(max(1, GENERATION_MIN_TOKENS_TO_KEEP)),
        generation_max_consecutive_zero_deltas=int(max(1, GENERATION_MAX_CONSECUTIVE_ZERO_DELTAS)),
        resume_from_checkpoint=resume_checkpoint,
        resume_mode=str(RESUME_MODE).lower(),
    )

    payload = {
        "profile": f"variant_{VARIANT}_{SIZE_PROFILE}",
        "model_profile": model_shape,
        "backend_status": backend_status,
        "data": {
            "manifest_path": str(manifest_out.resolve()),
            "train_pieces": int(len(train_manifest)),
            "val_pieces": int(len(val_manifest)),
            "source_npz_root": str(Path(PRETOKENIZED_ROOT).resolve()),
        },
        "training_profile": {
            "epochs": int(EPOCHS),
            "batch_size": int(BATCH_SIZE),
            "grad_accumulation_steps": int(GRAD_ACCUM_STEPS),
            "learning_rate": float(LEARNING_RATE),
            "warmup_steps": int(warmup_steps),
            "steps_per_epoch": int(steps_per_epoch),
            "total_steps": int(total_steps),
            "max_pieces": int(MAX_PIECES),
        },
        "result": result,
    }
    RESULT_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Training complete. Result JSON: {RESULT_JSON}")
else:
    print("Dry run only. Set RUN_TRAINING=True and rerun this cell to train.")

In [ ]:
import json

def print_result_summary(result_path: Path) -> None:
    if not result_path.exists():
        print(f"No result JSON yet: {result_path}")
        return

    payload = json.loads(result_path.read_text(encoding="utf-8"))
    result = payload.get("result", {})
    val_loss = result.get("val_loss", []) if isinstance(result, dict) else []

    print(f"Result path: {result_path}")
    print(f"Profile: {payload.get('profile')}")
    print(f"Params: {result.get('params')}")
    if val_loss:
        print(f"Best val loss: {min(val_loss):.6f}")
        print(f"Last val loss: {val_loss[-1]:.6f}")
    print(f"Output MIDI: {result.get('output_midi', '')}")
    print(f"Resume metadata: {result.get('resume', {})}")
    print(f"Backend status: {payload.get('backend_status', {})}")

print_result_summary(RESULT_JSON)

all_results = sorted((OUTPUT_BASE / "outputs").glob("sub100m_*_40m_*k/variant_*_40m_result.json"))
if all_results:
    print("\nKnown 40M run outputs:")
    for path in all_results[-10:]:
        print(f"  {path}")